# Load and process the dataset

Dataset: [Fraud Detection — 1M Transactions, 7 Fraud Types](https://www.kaggle.com/datasets/sergionefedov/fraud-detection-1m-transactions-7-fraud-types) (Kaggle).

This cell downloads the dataset, loads it into a `pandas.DataFrame`, and builds a scikit-learn preprocessing pipeline (imputation, scaling, one-hot encoding) so the result is ready to feed straight into a model in the next cell.

**Before running on a fresh machine:** set up a Kaggle API token (`~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME`/`KAGGLE_KEY` as Colab secrets) — see this folder's `README.md`.

In [ ]:
%pip install -q kagglehub scikit-learn pandas numpy

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import kagglehub
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Download the dataset and locate the CSV -------------------------------
dataset_dir = kagglehub.dataset_download(
    "sergionefedov/fraud-detection-1m-transactions-7-fraud-types"
)
csv_candidates = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)
assert csv_candidates, f"No CSV found under {dataset_dir}"
csv_path = csv_candidates[0]
print("Loading:", csv_path)

In [ ]:
# 2. Load and take a first look ---------------------------------------------
df = pd.read_csv(csv_path)
print("shape:", df.shape)
df.info()
df.head()

**Check the output above.** The next cell auto-detects the fraud label and the
column types from `df`'s dtypes/names — if the real column names differ from
what gets picked (print statements below make the guess visible), adjust
`target_col` / `id_like` accordingly before continuing.

In [ ]:
# 3. Identify the fraud label -------------------------------------------------
candidate_targets = [
    c for c in df.columns if any(k in c.lower() for k in ("fraud", "class", "label"))
]
print("Candidate target columns:", candidate_targets)
assert candidate_targets, "Could not auto-detect a fraud/label column — set target_col manually."

# Prefer the low-cardinality column (the binary is_fraud flag) over a
# free-text fraud-type column with 7+ categories.
target_col = min(candidate_targets, key=lambda c: df[c].nunique())
print("Using target column:", target_col)
print(df[target_col].value_counts(normalize=True))

In [ ]:
# 4. Basic cleaning ------------------------------------------------------------
df = df.drop_duplicates()

# Identifier-style columns carry no predictive signal and would just leak
# row identity through one-hot encoding — drop them.
id_like = [
    c for c in df.columns
    if c.lower() in ("transaction_id", "id", "customer_id", "account_id", "card_number")
]
print("Dropping id-like columns:", id_like)

y = df[target_col]
X = df.drop(columns=id_like + [target_col], errors="ignore")

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns")

In [ ]:
# 5. Preprocessing pipeline (scikit-learn) -------------------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

In [ ]:
# 6. Stratified train/test split + fit -----------------------------------------
# Fraud is a rare class, so stratify to keep the same fraud ratio in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit the preprocessor on train only, then transform test — avoids leaking
# test-set statistics (means/std/categories) into the transform.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed:", X_train_processed.shape)
print("X_test_processed: ", X_test_processed.shape)
print("train fraud rate:", y_train.mean() if y_train.dtype != object else y_train.value_counts(normalize=True))

`X_train_processed`, `X_test_processed`, `y_train`, `y_test` are now ready to
pass into `model.fit(...)` in the next cell. Class imbalance (e.g. SMOTE via
`imbalanced-learn`, or `class_weight="balanced"`) is a modeling-stage choice
left for that step.